In [8]:
from scripts.utils import get_config

config = get_config("band_map.json")

selected_arfcns

[648768, 643296, 643295, 641663, 641664, 638015, 638016]

In [1]:

import numpy as np

from scripts.beamforming import matrix_filter
from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import RF_PARAM_5G, NETWORK_TYPE

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt("data/random_seeds.csv", dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ["toa_pps", "toa_cir", "toa_cov", "campaign_id"]
df["measurements_matrix"] = df["measurements_matrix"].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

operator_choice = [10]

selected_campaigns = [1]
rf_param = RF_PARAM_5G.RSRQ

# Data filtering
df = filter_dataframe(
    df=df,
    operators=operator_choice,
    include_columns=[
        "pci",
        "beam_index",
        "nr_arfcn",
        "operator_id",
        "sinr",
        "rsrq"
    ],
    campaigns=selected_campaigns,
)




Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5


/Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/scripts/data_filter.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measurements_matrix"] = df["measurements_matrix"].apply(


In [16]:

mat = df.loc[6, "measurements_matrix"]

import pandas as pd


def matrix_filter(
        mat: pd.DataFrame,
        rf_param: RF_PARAM_5G,
        include_n_best_pcis: int = None,
        include_n_best_beams: int = None,
):
    """

    :param mat: Measurements matrix
    :param rf_param: What RF parameter to sort points by
    :param include_n_best_pcis: How many PCIs to include, set to None to include all PCI points
    :param include_n_best_beams: How many Beams to include, set to None to include all Beams
    :return:
    """

    mat = mat.dropna(subset=[rf_param.value])

    if mat.empty: return mat

    if not (include_n_best_pcis or include_n_best_beams):
        return mat
    # Get the maximum value for each PCI and beam_index combination
    idx = mat.groupby(['pci', 'beam_index'])[rf_param.value].idxmax()
    beams = mat.loc[idx].sort_values(by=[rf_param.value], ascending=False)

    # Filter for the n best PCIs if specified
    if include_n_best_pcis:
        # Get the n best unique PCIs based on their maximum RF parameter value
        best_pcis = beams.groupby('pci')[rf_param.value].max().nlargest(include_n_best_pcis).index.tolist()
        beams = beams[beams['pci'].isin(best_pcis)]

    # Filter for the b best beams for each PCI if specified
    if include_n_best_beams:
        # For each PCI, get the b best beams
        beams = beams.groupby('pci').apply(
            lambda x: x.nlargest(include_n_best_beams, rf_param.value)
        ).reset_index(drop=True)

    return beams


res1 = matrix_filter(
    mat=mat,
    rf_param=rf_param,
    include_n_best_pcis=1,
    include_n_best_beams=1
)

res2 = matrix_filter(
    mat=mat,
    rf_param=rf_param,
    include_n_best_pcis=1,
    include_n_best_beams=None
)

print(res1)
print('======')
print(res2)

res = mat[mat['pci'] == 75]
res = res[res['nr_arfcn'] == 643296]

res

   pci  beam_index  nr_arfcn  operator_id  sinr   rsrq
0   75           4    643296           10  2.61 -12.19
     pci  beam_index  nr_arfcn  operator_id   sinr   rsrq
132   75           4    643296           10   2.61 -12.19
124   75           0    643296           10   0.66 -12.99
130   75           3    643296           10   0.46 -13.06
126   75           1    643296           10  -3.57 -15.42
128   75           2    643296           10  -4.01 -15.73
135   75           7    643296           10  -9.05 -19.92
134   75           6    643296           10 -12.21 -22.74
133   75           5    643296           10 -13.58 -24.05


/var/folders/sz/wsd2tvj51v1c7jpt5gscdckc0000gn/T/ipykernel_96749/135689146.py:40: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  beams = beams.groupby('pci').apply(


,pci,beam_index,nr_arfcn,operator_id,sinr,rsrq
124,75,0,643296,10,0.66,-12.99
126,75,1,643296,10,-3.57,-15.42
128,75,2,643296,10,-4.01,-15.73
130,75,3,643296,10,0.46,-13.06
132,75,4,643296,10,2.61,-12.19
133,75,5,643296,10,-13.58,-24.05
134,75,6,643296,10,-12.21,-22.74
135,75,7,643296,10,-9.05,-19.92


In [ ]:
import pandas as pd
from scripts.utils import extract_unique_npcis


def matrix_filter(
        mat: pd.DataFrame,
        rf_param: RF_PARAM_5G,
        include_n_best_pcis: int = None,
        include_n_best_beams: int = None,
):
    """

    :param mat: Measurements matrix
    :param rf_param: What RF parameter to sort points by
    :param include_n_best_pcis: How many PCIs to include, set to None to include all PCI points
    :param include_n_best_beams: How many Beams to include, set to None to include all Beams
    :return:
    """

    mat = mat.dropna(subset=[rf_param.value])

    if mat.empty: return mat

    if not (include_n_best_pcis or include_n_best_beams):
        return mat
    # Get the maximum value for each PCI and beam_index combination
    idx = mat.groupby(['pci', 'beam_index'])[rf_param.value].idxmax()
    beams = mat.loc[idx].sort_values(by=[rf_param.value], ascending=False)

    # Filter for the n best PCIs if specified
    if include_n_best_pcis:
        # Get the n best unique PCIs based on their maximum RF parameter value
        best_pcis = beams.groupby('pci')[rf_param.value].max().nlargest(include_n_best_pcis).index.tolist()
        beams = beams[beams['pci'].isin(best_pcis)]

    # Filter for the b best beams for each PCI if specified
    if include_n_best_beams:
        # For each PCI, get the b best beams
        beams = beams.groupby('pci').apply(
            lambda x: x.nlargest(include_n_best_beams, rf_param.value)
        ).reset_index(drop=True)

    return beams


unique_before = extract_unique_npcis(df['measurements_matrix'])

# filter the df matricies

df.loc[:, "measurements_matrix"] = df.loc[:, "measurements_matrix"].apply(
    lambda x: matrix_filter(
        x,
        rf_param=RF_PARAM_5G.RSRQ,
        include_n_best_pcis=None,
        include_n_best_beams=1,
    )
)

unique_after = extract_unique_npcis(df['measurements_matrix'])

print(f'Before {len(unique_before)}, After {len(unique_after)}')

unique_before

In [ ]:
matrix[matrix['pci'] == 75]

In [ ]:
df['size'].sum()

In [ ]:
df['campaign_id'].unique().shape

In [ ]:
import folium
import pandas as pd


def geo_plot_points(df: pd.DataFrame):
    """
    Plots given locations to a map (OpenStreetMap) that is viewable in broswer.
    Generates a file called 'map.html' in the current working directory.
    :param df:
    """
    # Create a map centered around the mean location
    m = folium.Map(location=[df["lat"].mean(), df["lng"].mean()], zoom_start=12)

    # Add CircleMarkers to the map
    for _, row in df.iterrows():
        folium.CircleMarker(
            location=[row["lat"], row["lng"]],
            radius=1,  # Size of the marker
            color="blue",  # Border color of the marker
            fill=True,
            fill_color="blue",  # Fill color of the marker
            fill_opacity=0.6,
        ).add_to(m)

    # Save the map as an HTML file and open it in the browser
    m.save("map.html")


geo_plot_points(df.sample(3000))
